# WESAD — 02 · Windowing & Feature Extraction (wrist)  ·  v2

*Project: When Wearable Stress Models Fail — Subject Generalization and Distribution Shift.*

Turns the raw wrist signals into a **window-level feature matrix** — the artifact every downstream notebook
consumes. This version incorporates two findings from the first run:

- **Cardiac features are validity-gated.** Naive HRV on E4 wrist PPG was physiologically impossible
  (RMSSD ~250 ms). A signal-quality-gated detector brought that down but HRV stayed ~3× physiologic *even on
  the cleanest real windows* — this is a sensor limitation, not a code bug. **Decision: `hr_mean` is the sole
  trusted cardiac feature; `hr_std` and `hrv_*` are kept in the CSV as documented evidence but excluded from
  the model.** A per-window `ppg_quality` score is recorded.
- **HRV missingness is informative, not random.** HRV is NaN whenever PPG quality is poor (≈ half of windows),
  which tracks motion and may track condition. So we clean on **core** features only and *preserve* HRV NaNs —
  they are handled per-analysis in notebook 03, never imputed or dropped wholesale.

> Runs on Compute Canada (compute nodes are offline — install deps on the login node beforehand).
> Needs only `numpy pandas scipy matplotlib`. Self-contained. Writes the feature matrix to `outputs/tables/`.

## 1 · Setup (self-contained loader)

In [ ]:
from pathlib import Path
import pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks

warnings.filterwarnings("ignore", category=RuntimeWarning)

# >>> SET THIS to your WESAD root on Compute Canada <<<
DATA_DIR = Path("WESAD/")

SUBJECTS = [f"S{i}" for i in [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]]
FS = {"label": 700, "chest": 700, "wrist_ACC": 32, "wrist_BVP": 64,
      "wrist_EDA": 4, "wrist_TEMP": 4}
MODEL_STATES = {1: "baseline", 2: "stress", 3: "amusement"}

WINDOW_SEC = 60
SHIFT_SEC  = 60          # non-overlapping (honest default; nb 03 re-runs with overlap to show leakage)
PURITY     = 0.90
ACC_TO_G   = 64.0        # wrist ACC raw counts -> g

# Feature bookkeeping -----------------------------------------------------------
# GATED  : allowed to be NaN (HRV, quality-gated). Never used to drop windows; never imputed blindly.
# TRUSTED: the feature set the classifier is allowed to see (nb 03/05).
# ppg_quality is a covariate / confound probe, not a model input.
GATED_FEATURES   = ["hrv_sdnn", "hrv_rmssd", "hrv_pnn50"]
UNTRUSTED_EXTRA  = ["hr_std"] + GATED_FEATURES          # kept in CSV, excluded from model
TRUSTED_FEATURES = ["eda_mean","eda_std","eda_min","eda_max","eda_slope","eda_scr_count",
                    "temp_mean","temp_std","temp_min","temp_max","temp_slope",
                    "hr_mean",
                    "acc_std","acc_energy","acc_mad","acc_max"]

OUT = Path("outputs"); TAB = OUT/"tables"; FIG = OUT/"figures"
TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)


def load_subject(subj, data_dir=DATA_DIR):
    with open(Path(data_dir)/subj/f"{subj}.pkl", "rb") as f:
        return pickle.load(f, encoding="latin1")


def get_channel(data, location, channel):
    sig = np.asarray(data["signal"][location][channel], dtype=float)
    if channel == "ACC":
        sig = np.sqrt((sig**2).sum(axis=1))
        fs = FS[f"{location}_ACC"] if location == "wrist" else FS["chest"]
        if location == "wrist":
            sig = sig / ACC_TO_G
    else:
        sig = sig.ravel()
        fs = FS.get(f"{location}_{channel}", FS["chest"])
    return sig, fs

print("setup ok |", len(TRUSTED_FEATURES), "trusted features |",
      "untrusted/gated:", UNTRUSTED_EXTRA)

## 2 · Per-channel feature functions

EDA, TEMP, ACC as before. **BVP** uses the scipy-only, quality-gated cardiac extractor: prominence-gated
systolic peaks on a z-normalised band-passed signal, inter-beat intervals cleaned by physiologic bounds +
relative-median rejection. `hr_mean/hr_std` always returned; `hrv_*` only when the window's beats are clean,
else NaN. `ppg_quality` = fraction of plausible, artifact-free beats.

In [ ]:
def _bandpass(x, fs, lo, hi, order=3):
    ny = 0.5*fs; b, a = butter(order, [lo/ny, hi/ny], btype="band"); return filtfilt(b, a, x)


def eda_features(x, fs):
    x = np.asarray(x, float)
    if len(x) < 4:
        return dict.fromkeys(["eda_mean","eda_std","eda_min","eda_max","eda_slope","eda_scr_count"], np.nan)
    t = np.arange(len(x))/fs; coef = np.polyfit(t, x, 1)
    detr = x - np.polyval(coef, t)
    peaks, _ = find_peaks(detr, prominence=0.01)
    return dict(eda_mean=x.mean(), eda_std=x.std(ddof=1), eda_min=x.min(),
                eda_max=x.max(), eda_slope=coef[0], eda_scr_count=float(len(peaks)))


def temp_features(x, fs):
    x = np.asarray(x, float)
    if len(x) < 4:
        return dict.fromkeys(["temp_mean","temp_std","temp_min","temp_max","temp_slope"], np.nan)
    t = np.arange(len(x))/fs; slope = np.polyfit(t, x, 1)[0]
    return dict(temp_mean=x.mean(), temp_std=x.std(ddof=1), temp_min=x.min(),
                temp_max=x.max(), temp_slope=slope)


def bvp_features(x, fs):
    """E4 wrist PPG -> cardiac features, scipy-only, quality-gated (see notebook header)."""
    keys = ["hr_mean","hr_std","hrv_sdnn","hrv_rmssd","hrv_pnn50","ppg_quality"]
    x = np.asarray(x, float).ravel()
    if len(x) < fs*5 or np.allclose(x.std(), 0):
        return dict.fromkeys(keys, np.nan)
    try:
        xf = _bandpass(x, fs, 0.5, 4.0)
    except Exception:
        return dict.fromkeys(keys, np.nan)
    xf = (xf - xf.mean())/(xf.std() + 1e-9)
    peaks, _ = find_peaks(xf, distance=int(0.4*fs), prominence=0.5)
    if len(peaks) < 5:
        return dict.fromkeys(keys, np.nan)
    ibi = np.diff(peaks)/fs*1000.0
    ibi_ok = ibi[(ibi > 300) & (ibi < 1600)]
    if len(ibi_ok) < 4:
        return dict(hr_mean=np.nan, hr_std=np.nan, hrv_sdnn=np.nan,
                    hrv_rmssd=np.nan, hrv_pnn50=np.nan, ppg_quality=0.0)
    med = np.median(ibi_ok)
    clean = ibi_ok[np.abs(ibi_ok - med) < 0.30*med]
    quality = len(clean)/len(ibi)
    hr = 60000.0/ibi_ok
    out = dict(hr_mean=float(hr.mean()), hr_std=float(hr.std(ddof=1)), ppg_quality=float(quality))
    if quality >= 0.80 and len(clean) >= 4:
        out.update(hrv_sdnn=float(clean.std(ddof=1)),
                   hrv_rmssd=float(np.sqrt(np.mean(np.diff(clean)**2))),
                   hrv_pnn50=float(np.mean(np.abs(np.diff(clean)) > 50)*100))
    else:
        out.update(hrv_sdnn=np.nan, hrv_rmssd=np.nan, hrv_pnn50=np.nan)
    return out


def acc_features(mag, fs):
    mag = np.asarray(mag, float)
    if len(mag) < 4:
        return dict.fromkeys(["acc_std","acc_energy","acc_mad","acc_max"], np.nan)
    d = np.abs(np.diff(mag))
    return dict(acc_std=mag.std(ddof=1), acc_energy=float(np.mean((mag - mag.mean())**2)),
                acc_mad=float(d.mean()), acc_max=mag.max())

print("feature functions defined")

## 3 · Windowing with label-purity gating

A window gets a label only if one affect state covers >= `PURITY` of the whole window; boundary-straddling
windows are dropped. Each channel is sliced on its own timebase.

In [ ]:
def window_label(data, t0, t1, purity=PURITY):
    lab = np.asarray(data["label"]).ravel()
    i0, i1 = int(t0*FS["label"]), int(t1*FS["label"])
    seg = lab[i0:i1]
    if len(seg) == 0:
        return None
    affect = seg[np.isin(seg, list(MODEL_STATES))]
    if len(affect) == 0:
        return None
    vals, counts = np.unique(affect, return_counts=True)
    top = int(vals[counts.argmax()])
    if counts.max()/len(seg) < purity:
        return None
    return top


def extract_subject(subj, window_sec=WINDOW_SEC, shift_sec=SHIFT_SEC):
    d = load_subject(subj)
    dur = len(np.asarray(d["label"]))/FS["label"]
    chans = {ch: get_channel(d, "wrist", ch) for ch in ["EDA", "TEMP", "BVP", "ACC"]}
    rows, t0 = [], 0.0
    while t0 + window_sec <= dur:
        t1 = t0 + window_sec
        lab = window_label(d, t0, t1)
        if lab is not None:
            rec = dict(subject=subj, condition=lab, condition_name=MODEL_STATES[lab], t_start=round(t0, 3))
            for ch, (sig, fs) in chans.items():
                seg = sig[int(t0*fs):int(t1*fs)]
                if ch == "EDA":   rec.update(eda_features(seg, fs))
                elif ch == "TEMP": rec.update(temp_features(seg, fs))
                elif ch == "BVP":  rec.update(bvp_features(seg, fs))
                elif ch == "ACC":  rec.update(acc_features(seg, fs))
            rows.append(rec)
        t0 += shift_sec
    return rows

print("windowing defined")

## 4 · Extract all subjects

In [ ]:
all_rows = []
for s in SUBJECTS:
    r = extract_subject(s); all_rows.extend(r)
    print(f"{s}: {len(r):4d} windows")

feat = pd.DataFrame(all_rows)
print("\nTotal windows:", len(feat))
print("\nClass balance:"); print(feat["condition_name"].value_counts())

### 4.1 · Cleaning — drop on CORE features only, PRESERVE HRV NaNs

This is the critical step. HRV (`hrv_*`) is NaN by design on low-quality windows (~half of them), so dropping
any row with *any* NaN would delete half the data and distort class balance. We drop only on **core** features
(everything except the quality-gated HRV) and keep HRV NaNs for per-analysis handling in notebook 03.

In [ ]:
core = [c for c in feat.columns
        if c not in {"subject", "condition", "condition_name", "t_start"}
        and c not in GATED_FEATURES]

na_core = feat[core].isna().mean().sort_values(ascending=False)
print("NaN rate, CORE features (expected ~0):")
print(na_core[na_core > 0] if (na_core > 0).any() else "  all zero")

before = len(feat)
feat_clean = feat.dropna(subset=core).reset_index(drop=True)   # HRV NaNs preserved
print(f"\nDropped {before - len(feat_clean)}/{before} windows on missing CORE features "
      f"({100*(before-len(feat_clean))/before:.1f}%)")

hrv_avail = feat_clean["hrv_rmssd"].notna().mean()
print(f"HRV available on {hrv_avail*100:.0f}% of kept windows "
      f"(NaN elsewhere BY DESIGN -- not dropped, not imputed here)")

print("\nClass balance after cleaning:")
print(feat_clean["condition_name"].value_counts())
print("\nWindows per subject (should be ~30-36 each):")
print(feat_clean.groupby("subject").size().reindex(SUBJECTS))

## 5 · Save the feature matrix

Full matrix (HRV NaNs preserved) plus a provenance record and the trusted/untrusted feature lists, so
downstream notebooks model on the right columns without guesswork.

In [ ]:
stamp = f"w{WINDOW_SEC}_s{SHIFT_SEC}"
csv_path = TAB/f"wrist_features_{stamp}.csv"
feat_clean.to_csv(csv_path, index=False)
print("wrote", csv_path, "shape", feat_clean.shape)

pd.Series(dict(window_sec=WINDOW_SEC, shift_sec=SHIFT_SEC, purity=PURITY,
               n_windows=len(feat_clean),
               hrv_available_frac=round(float(feat_clean["hrv_rmssd"].notna().mean()), 3),
               subjects=len(SUBJECTS))).to_csv(TAB/f"feature_provenance_{stamp}.csv")

pd.DataFrame({"feature": TRUSTED_FEATURES + UNTRUSTED_EXTRA,
              "trusted": [True]*len(TRUSTED_FEATURES) + [False]*len(UNTRUSTED_EXTRA)}
             ).to_csv(TAB/"feature_trust.csv", index=False)
print("wrote feature_trust.csv (trusted vs untrusted/gated)")
feat_clean.head()

## 6 · Sanity checks

**(a)** Do trusted features separate the states *within* subject (z-scored per subject to remove the offset)?
**(b)** Early confound peek: does `ppg_quality` differ by condition? If stress windows have lower PPG quality
(more motion), then signal quality — and HRV missingness — is confounded with the label. Flag it here; test it
properly in notebook 03.

In [ ]:
plot_feats = ["eda_mean", "temp_slope", "hr_mean", "eda_scr_count", "acc_std"]
plot_feats = [f for f in plot_feats if f in feat_clean.columns]
order = ["baseline", "stress", "amusement"]

z = feat_clean.copy()
z[plot_feats] = z.groupby("subject")[plot_feats].transform(lambda c: (c - c.mean())/(c.std(ddof=0) + 1e-9))

fig, axes = plt.subplots(1, len(plot_feats) + 1, figsize=(3*(len(plot_feats)+1), 3.8))
for ax, f in zip(axes[:-1], plot_feats):
    ax.boxplot([z.loc[z.condition_name == c, f].values for c in order], tick_labels=order, showfliers=False)
    ax.set_title(f, fontsize=10); ax.tick_params(axis="x", labelsize=8, rotation=20)
axes[0].set_ylabel("within-subject z")

# (b) ppg_quality by condition -- RAW (not z-scored): is signal quality confounded with label?
axes[-1].boxplot([feat_clean.loc[feat_clean.condition_name == c, "ppg_quality"].values for c in order],
                 tick_labels=order, showfliers=False)
axes[-1].set_title("ppg_quality by condition\n(confound peek)", fontsize=9)
axes[-1].tick_params(axis="x", labelsize=8, rotation=20)

fig.suptitle("Trusted features separate state within subject | last panel: is PPG quality confounded with condition?")
plt.tight_layout(); plt.savefig(FIG/"within_subject_feature_separation.png", dpi=140); plt.show()

q_by_c = feat_clean.groupby("condition_name")["ppg_quality"].agg(["mean", "median", "count"]).reindex(order)
print("PPG quality by condition:"); print(q_by_c)

## 7 · Handoff to notebook 03

**Artifacts** (upload these back): `wrist_features_w60_s60.csv` (full matrix, HRV NaNs preserved),
`feature_provenance_w60_s60.csv`, `feature_trust.csv`, and the figure.

**Feature policy carried forward:** model on `TRUSTED_FEATURES` (16 cols: EDA, TEMP, HR-mean, ACC).
`hr_std` + `hrv_*` stay in the CSV as documented evidence of the E4-HRV limitation but are **not** modelled.
`ppg_quality` is a covariate / confound probe, not a feature.

**Notebook 03 (the five pillars):**
1. **Leakage audit** — naive random split + global scaling + overlapping windows (regen at shift = 0.25 s)
   → inflated; then LOSO + fold-fit scaling → honest number. Report the gap.
2. **Confounding probes** — (i) drop `acc_*`, re-evaluate: motion detector or stress detector? (ii) test
   whether `ppg_quality` / HRV-missingness differs by condition (data-quality confound).
3. **LOSO with confidence intervals** — 15 folds × seeds; per-subject spread; CI on the generalization gap.
4. **Calibration / uncertainty** — reliability curves, Brier score, optional conformal sets.
5. **What to do next** — connect the cross-subject result to Eli's *longitudinal within-subject* problem.